In [ ]:
!pip install catboost

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import kagglehub
import os

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRegressor




In [ ]:


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:

df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:

df = df.drop(columns='Order_ID')
df.columns

In [ ]:
# Task 2: Write your code here:
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Percentage': missing_percentage.values,
    'Data_Type': df.dtypes.values
})

missing_data = (
    missing_data[missing_data['Missing_Percentage'] > 0]
    .sort_values('Missing_Percentage', ascending=False)
)
missing_data.head(10)




# this codes i find it before exam to auto handling missing values :)
cols_to_fill = missing_data.loc[(missing_data['Missing_Percentage'] < 50) &(missing_data['Column'].isin(df.columns)),'Column']

# Numeric columns → median
numeric_cols = df[cols_to_fill].select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Object columns → mode
object_cols = df[cols_to_fill].select_dtypes(include=['object']).columns
for col in object_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# RE-CHECK
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Percentage': missing_percentage.values,
    'Data_Type': df.dtypes.values
})

missing_data = (
    missing_data[missing_data['Missing_Percentage'] > 0]
    .sort_values('Missing_Percentage', ascending=False)
)
missing_data.head(10)

# Missing values handled

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

#Check Categorical Columns
def encode_categorical_columns(df):
    categorical_cols = df.select_dtypes(include=["object"]).columns
    print("Categorical Columns:", list(categorical_cols))

label_encoders = encode_categorical_columns(df)

# Transfer all objects to new


categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

In [ ]:
# Task 5: Write your code here:


numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 6: Write your code here:

# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time",axis=1)
y = df['Delivery_Time']

X.head()


In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []
predictions = []
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

for train_idx, val_idx in kfold.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train and predict
    model.fit(X_train, y_train)
    y_fold_pred = model.predict(X_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_val, y_fold_pred))
    predictions.append(y_fold_pred)


mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols = X.columns
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.hist(predictions, bins=25)
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Freq")
plt.show()

In [ ]:
# Task Bonus: Write your code here:
mea_avg = []

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []
predictions = []
model_1 = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model_2 = CatBoostRegressor(verbose=0, random_state=42)

for train_idx, val_idx in kfold.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train and predict
    model_1.fit(X_train, y_train)
    model_2.fit(X_train, y_train)
    y_fold_pred1 = model_1.predict(X_val)
    y_fold_pred2 = model_2.predict(X_val)

    firstloss = mean_absolute_error(y_val, y_fold_pred1)
    secendloss = mean_absolute_error(y_val, y_fold_pred2)
    # Calculate metrics
    mea_avg.append((firstloss + secendloss) / 2)


mea_avg = np.array(mea_avg)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mea_avg.mean():,.2f}")
